# EfficientNetB0: final held-out test evaluation

**Status: unexecuted.** Training and model selection are complete. Evaluate the selected partial-fine-tuning checkpoint (epoch 4) without changing weights. Use the existing shared test manifest and evaluation transforms. Do not tune settings using these test results.

Run cells in order with the same project Python kernel. The trained checkpoint and images must be present locally. This notebook does not train or download pretrained weights. Results are saved separately from both training runs.

## 1. Locate the project and import dependencies

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, platform, subprocess, time
import numpy as np
import pandas as pd
import yaml
import torch
import torchvision
import matplotlib.pyplot as plt
from IPython.display import display
from torch.utils.data import DataLoader
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'src/data/dataset.py').is_file()
                     and (p / 'configs/base.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook inside waste-classification.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data.dataset import WasteDataset
from src.data.transforms import get_eval_transforms
from src.models.efficientnetb0 import create_efficientnetb0
from src.evaluation.metrics import collect_predictions, calculate_classification_metrics

base_config = yaml.safe_load((PROJECT_ROOT / 'configs/base.yaml').read_text())
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print('Project:', PROJECT_ROOT)
print('Device:', device)


## 2. Verify the selected checkpoint and unchanged experiment inputs

In [ ]:
RUN_NAME = 'efficientnetb0_partial_finetuning_20260927T010052451538Z'
TRAINING_RUN = PROJECT_ROOT / 'results/efficientnetb0' / RUN_NAME
checkpoint_path = TRAINING_RUN / 'best.pth'
assert checkpoint_path.is_file(), f'Missing checkpoint: {checkpoint_path}'
selection_path = TRAINING_RUN / 'selection.json'
assert selection_path.is_file(), 'Run section 9 of notebook 08 first to record model selection.'
selection = json.loads(selection_path.read_text())
assert (PROJECT_ROOT / selection['selected_checkpoint']).resolve() == checkpoint_path.resolve()
metadata = json.loads((TRAINING_RUN / 'run_metadata.json').read_text())
assert metadata['training_complete']
class_to_idx = json.loads((TRAINING_RUN / 'class_to_idx.json').read_text())
class_names = sorted(class_to_idx, key=class_to_idx.get)
assert class_names == base_config['classes']
MANIFEST_DIR = PROJECT_ROOT / base_config['data']['manifest_dir']
DATASET_ROOT = Path(os.environ.get('WASTE_DATASET_ROOT',
                    str(PROJECT_ROOT / base_config['data']['dataset_root']))).expanduser()
for split, expected_hash in metadata['manifest_sha256'].items():
    assert hashlib.sha256((MANIFEST_DIR / f'{split}.csv').read_bytes()).hexdigest() == expected_hash, f'{split} manifest changed.'
for name in ['src/data/transforms.py', 'src/models/efficientnetb0.py']:
    assert hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest() == metadata['source_file_sha256'][name], f'Source changed: {name}'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
assert checkpoint['class_to_idx'] == class_to_idx
assert int(checkpoint['epoch']) == 4, 'Expected the selected epoch 4 checkpoint.'
print('Selected checkpoint:', checkpoint_path)
print('Selected epoch:', checkpoint['epoch'])
print('Selection validation loss:', checkpoint['val_loss'])


## 3. Load the held-out test data and restore the model

In [ ]:
test_frame = pd.read_csv(MANIFEST_DIR / 'test.csv')
assert len(test_frame) == 2965, 'Unexpected test set size; check the shared manifests.'
assert not test_frame[['relative_path', 'class_name', 'split']].isna().any().any()
assert test_frame['relative_path'].is_unique
assert test_frame['split'].eq('test').all()
assert set(test_frame['class_name']) == set(class_names)
for split in ['train', 'val']:
    other = pd.read_csv(MANIFEST_DIR / f'{split}.csv')
    assert not set(test_frame['relative_path']).intersection(other['relative_path'])
missing = [s for s in test_frame['relative_path'] if not (DATASET_ROOT / s).is_file()]
assert not missing, f'Missing test images: {missing[:5]}'
# Path checks do not establish that visually similar images are split independently.
test_dataset = WasteDataset(MANIFEST_DIR / 'test.csv', DATASET_ROOT,
                            transform=get_eval_transforms(), class_to_idx=class_to_idx)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,
                         num_workers=0, pin_memory=(device.type == 'cuda'))
model = create_efficientnetb0(num_classes=len(class_names), pretrained=False,
                              freeze_backbone=True)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.requires_grad_(False)
model.to(device).eval()
print('Test images:', len(test_dataset))
print('Classes:', class_names)
print('Model restored; all gradient updates disabled.')


## 4. Evaluate and save the actual test results

This runs prediction only. If a completed report already exists, the cell stops to avoid overwriting it. Training results and model weights stay unchanged.

In [ ]:
EVAL_DIR = PROJECT_ROOT / 'results/efficientnetb0' / (RUN_NAME + '_final_test')
EVAL_DIR.mkdir(parents=True, exist_ok=True)
assert not (EVAL_DIR / 'test_metrics.json').exists(), 'Test results already exist. Read the saved report rather than overwrite it.'
start = time.perf_counter()
y_true, y_pred, y_prob = collect_predictions(model, test_loader, device)
elapsed = time.perf_counter() - start
expected_labels = test_frame['class_name'].map(class_to_idx).to_numpy()
assert np.array_equal(y_true, expected_labels), 'Prediction order does not match the manifest.'
assert y_prob.shape == (len(test_frame), len(class_names))
assert np.isfinite(y_prob).all()
metrics, report, matrix = calculate_classification_metrics(y_true, y_pred, class_names)
assert int(matrix.sum()) == len(test_frame)
(EVAL_DIR / 'test_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
pd.DataFrame(report).T.to_csv(EVAL_DIR / 'test_classification_report.csv')
pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(EVAL_DIR / 'test_confusion_matrix.csv')
predictions = test_frame[['relative_path', 'class_name']].copy()
predictions['predicted_class'] = [class_names[int(i)] for i in y_pred]
predictions['top_class_score'] = y_prob.max(axis=1)
predictions.to_csv(EVAL_DIR / 'test_predictions.csv', index=False)
(EVAL_DIR / 'class_to_idx.json').write_text(json.dumps(class_to_idx, indent=2) + '\n')
try:
    source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
except (OSError, subprocess.CalledProcessError):
    source_commit = None
evaluation_metadata = {
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'checkpoint': str(checkpoint_path.relative_to(PROJECT_ROOT)),
    'checkpoint_sha256': hashlib.sha256(checkpoint_path.read_bytes()).hexdigest(),
    'selected_epoch': int(checkpoint['epoch']),
    'selection': selection,
    'test_count': len(test_frame),
    'manifest_sha256': metadata['manifest_sha256'],
    'class_to_idx': class_to_idx,
    'source_commit': source_commit,
    'device': str(device), 'python': platform.python_version(),
    'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
    'evaluation_seconds': elapsed,
    'preprocessing': 'shared get_eval_transforms; shuffle=False; batch_size=32',
    'source_sha256': {name: hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest()
        for name in ['src/data/dataset.py', 'src/data/transforms.py',
                     'src/models/efficientnetb0.py', 'src/evaluation/metrics.py']},
}
(EVAL_DIR / 'evaluation_metadata.json').write_text(json.dumps(evaluation_metadata, indent=2) + '\n')
print('Test evaluation seconds:', round(elapsed, 1))
print('Correct:', int((y_true == y_pred).sum()), '/', len(y_true))
display(pd.DataFrame([metrics]))
display(pd.DataFrame(report).T.round(4))


## 5. Plot the confusion matrix and place validation and test metrics side by side

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9), constrained_layout=True)
ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(
    ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
ax.set_title('EfficientNetB0: held-out test confusion matrix')
fig.savefig(EVAL_DIR / 'test_confusion_matrix.png', dpi=150)
plt.show()
validation_metrics = json.loads((TRAINING_RUN / 'validation_metrics.json').read_text())
comparison = pd.DataFrame([
    {'split': 'Validation (used for selection)', **validation_metrics},
    {'split': 'Test (final evaluation)', **metrics},
])
comparison.to_csv(EVAL_DIR / 'validation_test_comparison.csv', index=False)
display(comparison)
print('Results folder:', EVAL_DIR)
print('The trained checkpoint stays in its original training folder; keep its Drive backup.')


## 6. Interpret and save

Replace these prompts with your actual outputs:

- State test accuracy and macro F1, and the number of test images.
- Identify the lowest-recall and lowest-F1 classes.
- Describe the largest off-diagonal confusion counts.
- Compare validation and test results without using test results to tune this model.
- Note that this is one split and one training seed. Document any unresolved near-duplicate or label-quality issues with the team; high scores alone do not establish real-world performance.

Save this notebook with its outputs. Commit it and the final-test report folder. Back up the new report folder and retain the existing trained checkpoint backup. No new model checkpoint is created by evaluation.
